# Notebook A: what is inside the cached LLaVA residual streams? (CPU only, ~10 min)

**Question this answers.** In the counts10 run the model's Right/Left lean $\Delta$ shifts when the prompt says e.g. "In the last 10 trials the grating was on the left 2 times and on the right 8 times". Probes said this is *readable* in the residual stream at the answer position. This notebook asks whether the readable thing is the thing the behavior follows:

1. **A1.** Is there a linear direction (the *count direction*) in the hist-run residual stream that tracks the number of "right" in the prompt? Out-of-fold, per layer.
2. **A2.** Does the position along that direction predict each trial's behavioral shift $\Delta_{hist}-\Delta_{none}$ *beyond* the count itself? (If yes, the direction is not just a copy of the number; it carries something the readout uses, e.g. image-dependent gain.)
3. **A3.** Is the representation graded in the count, while behavior is flat for 1-7 "right" and jumps at 8-10? (Tells us whether the nonlinearity is in the representation or in the readout.)
4. **A4.** Geometry: is the count direction aligned with or orthogonal to the grating-side (evidence) direction? Does the evidence signal already shrink inside the residual stream when history is in the prompt (a representation-level version of the $\beta$ suppression), and at which layer?
5. Saves the directions as small `.npy` files for the later GPU patching/steering notebook.

**Setup.** Add the dataset `llava-pve-outputs-run` as input (already there if the path below exists). No GPU, no internet needed. Use *Save & Run All (Commit)*.

Caution about the design: the counts10 prompt contains **only the count** of the last 10 trials, not their order and not the block. So the model cannot know the block beyond what the count implies. "Count" is therefore the right variable here, and the earlier "text beats block" comparison was expected by construction. What this notebook adds is what the *network does* with the count.

In [ ]:
# ===================== CONFIG =====================
MOCK = False
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
OUT  = "/kaggle/working/A_out"
FOLDS = 5
N_BOOT = 300
PEEK_LAYERS = [4, 8, 12, 16, 20, 24, 28, 32]     # for the graded-ness plot (A3)
import os, glob, math, time, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""):
    print(s); SUMMARY.append(s)

In [ ]:
# ===================== LOAD =====================
if MOCK:
    rng = np.random.default_rng(1); N, NL, D = 600, 33, 128
    side = rng.choice(["left", "right"], N); contrast = rng.choice([0.0, 0.03, 0.045, 0.0625, 0.09, 0.125, 1.0], N)
    p = np.repeat(rng.choice([0.2, 0.8], 40), 15)[:N]; p[:90] = 0.5
    nh = np.minimum(np.arange(N), 10); nr = np.array([rng.binomial(n, 0.8 if (pp == 0.8) else 0.2 if pp == 0.2 else 0.5) for n, pp in zip(nh, p)])
    sgn = np.where(side == "right", 1, -1) * (contrast > 0)
    tp_ = np.log(((nr + .5) / (nh + 1)) / (1 - (nr + .5) / (nh + 1)))
    gain = 1 + 0.3 * rng.normal(size=N)
    d_none = 15 * sgn * np.minimum(contrast, 0.12) + rng.normal(0, .3, N)
    d_hist = 0.5 * d_none + 0.4 * gain * (nr >= 8) * 1.0 + 0.1 * tp_ * (nh == 10) + rng.normal(0, .3, N)
    ROWS = pd.concat([pd.DataFrame(dict(t=np.arange(N), order=o, side=side, contrast=contrast, p_right=p,
                                        n_hist=np.where(o == "none", 0, nh), n_hist_right=np.where(o == "none", 0, nr), delta=d))
                      for o, d in (("none", d_none), ("hist", d_hist))], ignore_index=True)
    ROWS.loc[ROWS.order == "none", ["n_hist", "n_hist_right"]] = np.c_[nh, nr]        # metadata identical in both conditions, as in the real file
    HS = rng.normal(size=(N, 2, NL, D)).astype(np.float16)
    for L in range(4, NL):
        HS[:, 1, L, 0] += 2 * tp_ * (nh == 10); HS[:, :, L, 1] += 3 * sgn[:, None] * (contrast[:, None] > 0.05) * np.array([[1, .6]])
else:
    files = sorted(glob.glob(f"{DATA}/**/*", recursive=True))
    print(f"{len(files)} files under {DATA}"); [print("  ", f.replace(DATA, "."), f"{os.path.getsize(f)/1e6:.1f} MB") for f in files if os.path.isfile(f)][:60]
    hsf = sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))
    ra = glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)
    assert hsf, "No full_hs_*.npy found: the residual-stream cache is not in this dataset. Tell Claude; we rerun the full pass (SEED=0 reproduces the trials)."
    if ra: ROWS = pd.read_csv(ra[0])
    else:  ROWS = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{DATA}/**/full_rows_0*.csv", recursive=True))], ignore_index=True)
    HS = np.concatenate([np.load(f, mmap_mode="r") for f in hsf])          # (trial, order[none,hist], layer, dim) float16
    print("HS", HS.shape, HS.dtype, "| chunks", len(hsf), "| rows", ROWS.shape)

w = ROWS.pivot(index="t", columns="order", values="delta")
M = ROWS[ROWS.order == "none"].set_index("t").loc[w.index][["side", "contrast", "p_right", "n_hist", "n_hist_right"]].copy()
M["d_none"], M["d_hist"] = w["none"], w["hist"]; M["shift"] = M.d_hist - M.d_none
assert len(M) == HS.shape[0] and (M.index.values == np.arange(len(M))).all(), "trial index must match cache order"
NL = HS.shape[2]
pr = M.p_right.round(2).values; blk = np.concatenate([[0], np.cumsum(pr[1:] != pr[:-1])])
FULL = (M.n_hist.values == 10)                                    # trials with a full 10-trial window
frac = (M.n_hist_right.values + .5) / (M.n_hist.values + 1); TP = np.log(frac / (1 - frac)); NR = M.n_hist_right.values.astype(float)
cell = pd.get_dummies(M.side.astype(str) + "|" + M.contrast.astype(str), dtype=float).values   # side x contrast cells
print(f"trials {len(M)} | full-window trials {FULL.sum()} | blocks {blk.max()+1}")

In [ ]:
# ===================== helpers =====================
def lsq(C, Y): return np.linalg.lstsq(C, Y, rcond=None)[0]

def oof_direction_projection(X, target, C, groups):
    """Out-of-fold projection on the 'target direction': fit cell means and direction on train blocks, project held-out blocks."""
    z = np.full(len(X), np.nan)
    for tr, te in GroupKFold(FOLDS).split(X, groups=groups):
        b = lsq(C[tr], X[tr]); Xtr, Xte = X[tr] - C[tr] @ b, X[te] - C[te] @ b
        yt = target[tr] - C[tr] @ lsq(C[tr], target[tr])
        d = Xtr.T @ yt; d /= (np.linalg.norm(d) + 1e-12)
        z[te] = Xte @ d
    return z

def full_direction(X, target, C):
    b = lsq(C, X); Xr = X - C @ b; yt = target - C @ lsq(C, target)
    d = Xr.T @ yt; return d / (np.linalg.norm(d) + 1e-12)

def cv_r2(y, Z, groups):
    pred = np.zeros_like(y)
    for tr, te in GroupKFold(FOLDS).split(Z, groups=groups):
        pred[te] = Z[te] @ lsq(Z[tr], y[tr])
    return 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()

def partial_corr(a, b, C):
    ra, rb = a - C @ lsq(C, a), b - C @ lsq(C, b)
    return float(np.corrcoef(ra, rb)[0, 1])

def block_boot_ci(fn, groups, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed); ug = np.unique(groups); idx_by = {g: np.where(groups == g)[0] for g in ug}; vals = []
    for _ in range(n):
        pick = rng.choice(ug, len(ug)); idx = np.concatenate([idx_by[g] for g in pick])
        try: vals.append(fn(idx))
        except Exception: pass
    return np.nanpercentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)

In [ ]:
# ===================== A1 + A2: count direction per layer, and does it predict behavior? =====================
m = FULL
C = cell[m]; g = blk[m]; y_shift = M["shift"].values[m]; tp = TP[m]; nr = NR[m]
Cb = np.c_[C, tp]                     # baseline for the shift: cells + count
rows_a = []
t0 = time.time()
for L in range(NL):
    rec = dict(layer=L)
    for oi, o in enumerate(["hist", "none"]):
        X = HS[:, 1 if o == "hist" else 0, L, :][m].astype(np.float32)
        z = oof_direction_projection(X, tp, C, g)                       # count direction, out of fold
        rec[f"count_corr_{o}"] = partial_corr(z, tp, C)                  # how well the direction tracks the count (cells removed)
        if o == "hist":
            rec["partial_corr_shift"] = partial_corr(z, y_shift, Cb)     # does z explain the shift beyond cells + count?
            r_base = cv_r2(y_shift, Cb, g); r_full = cv_r2(y_shift, np.c_[Cb, z], g)
            rec["cvR2_base"], rec["cvR2_with_z"], rec["dR2"] = r_base, r_full, r_full - r_base
            lo, hi = block_boot_ci(lambda idx: partial_corr(z[idx], y_shift[idx], Cb[idx]), g, n=120)
            rec["pc_lo"], rec["pc_hi"] = lo, hi
            # block-label AUROC of z within side (comparison with earlier probe numbers)
            pb = M.p_right.round(2).values[m]; au = []
            for sd in ("left", "right"):
                mm = (M.side.values[m] == sd) & (pb != 0.5)
                if mm.sum() > 20 and len(np.unique(pb[mm])) == 2: au.append(roc_auc_score(pb[mm] == 0.8, z[mm]))
            rec["block_auc"] = float(np.mean(au)) if au else np.nan
    rows_a.append(rec)
    if L % 8 == 0: print(f"layer {L} done ({time.time()-t0:.0f}s)", flush=True)
A1 = pd.DataFrame(rows_a); A1.to_csv(f"{OUT}/a1_a2_layers.csv", index=False)
say("[A1] count direction (out of fold, trials with full 10-window): corr with count after removing side x contrast cells")
say(A1[["layer", "count_corr_hist", "count_corr_none", "block_auc"]].iloc[::2].round(2).to_string(index=False))
say("\n[A2] does the projection explain the per-trial behavioral shift beyond cells + count?  (partial corr; dR2 = out-of-fold R2 gain)")
say(A1[["layer", "partial_corr_shift", "pc_lo", "pc_hi", "cvR2_base", "dR2"]].iloc[::2].round(3).to_string(index=False))

In [ ]:
# ===================== A3: graded-ness in the count (representation) vs behavior =====================
nr_i = NR[m].astype(int); rowsg = []
beh = pd.Series(y_shift).groupby(nr_i).mean()
for L in PEEK_LAYERS:
    if L >= NL: continue
    X = HS[:, 1, L, :][m].astype(np.float32); z = oof_direction_projection(X, tp, C, g)
    zr = z - C @ lsq(C, z)                                               # cells removed
    zz = pd.Series(zr).groupby(nr_i).mean()
    onehot = pd.get_dummies(nr_i, dtype=float).values; lin = np.c_[np.ones(len(nr_i)), nr_i]
    def r2(Z, y): res = y - Z @ lsq(Z, y); return 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
    rowsg.append(dict(layer=L, R2_linear_in_count=r2(lin, zr), R2_free_per_count=r2(onehot, zr),
                      R2_linear_beh=r2(lin, y_shift), R2_free_beh=r2(onehot, y_shift)))
A3 = pd.DataFrame(rowsg); A3.to_csv(f"{OUT}/a3_graded.csv", index=False)
say("\n[A3] how linear in the count are the representation (z) and the behavior (shift)?  R2 of a straight line vs a free value per count")
say(A3.round(3).to_string(index=False))
say("behavior: mean shift by number of 'right' in the prompt: " + str(beh.round(2).to_dict()))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for L in PEEK_LAYERS:
    if L >= NL: continue
    X = HS[:, 1, L, :][m].astype(np.float32); z = oof_direction_projection(X, tp, C, g); zr = z - C @ lsq(C, z)
    s = pd.Series(zr).groupby(nr_i).mean(); ax[0].plot(s.index, (s - s.mean()) / (s.std() + 1e-9), marker="o", ms=3, label=f"L{L}")
ax[0].set_title("representation: mean projection on count direction, by count (z-scored)"); ax[0].set_xlabel("# right in prompt (of 10)"); ax[0].legend(fontsize=7)
ax[1].plot(beh.index, beh.values, "k-o"); ax[1].set_title("behavior: mean (Δ_hist − Δ_none) by count"); ax[1].set_xlabel("# right in prompt (of 10)"); ax[1].axhline(0, c="gray", lw=.5)
plt.tight_layout(); plt.savefig(f"{OUT}/a3_graded.png", dpi=130); plt.close()

In [ ]:
# ===================== A4: geometry; does the evidence signal shrink inside the residual stream? =====================
hi_c = (M.contrast.values >= 0.09); sr = (M.side.values == "right")
rows_g = []; PRIOR_DIR, EVID_DIR, SHIFT_DIR = [], [], []
for L in range(NL):
    Xn = HS[:, 0, L, :].astype(np.float32); Xh = HS[:, 1, L, :].astype(np.float32)
    e_n = Xn[hi_c & sr].mean(0) - Xn[hi_c & ~sr].mean(0); e_h = Xh[hi_c & sr].mean(0) - Xh[hi_c & ~sr].mean(0)
    p_dir = full_direction(Xh[m], tp, C)
    hi, lo = (NR >= 8) & FULL, (NR <= 2) & FULL
    s_dir = (Xh - Xn)[hi].mean(0) - (Xh - Xn)[lo].mean(0)
    un = e_n / (np.linalg.norm(e_n) + 1e-12)
    def dprime(X):
        a, b = X[hi_c & sr] @ un, X[hi_c & ~sr] @ un
        return (a.mean() - b.mean()) / (0.5 * (a.std() + b.std()) + 1e-9)
    rows_g.append(dict(layer=L, cos_prior_evid=float(p_dir @ un), cos_evid_none_hist=float(e_n @ e_h / (np.linalg.norm(e_n) * np.linalg.norm(e_h) + 1e-12)),
                       dprime_none=dprime(Xn), dprime_hist=dprime(Xh), evid_norm_ratio=float(np.linalg.norm(e_h) / (np.linalg.norm(e_n) + 1e-12)),
                       cos_prior_vs_shiftdir=float(p_dir @ s_dir / (np.linalg.norm(s_dir) + 1e-12))))
    PRIOR_DIR.append(p_dir); EVID_DIR.append(un); SHIFT_DIR.append(s_dir)
A4 = pd.DataFrame(rows_g); A4["dprime_ratio"] = A4.dprime_hist / A4.dprime_none; A4.to_csv(f"{OUT}/a4_geometry.csv", index=False)
np.save(f"{OUT}/prior_dir.npy", np.array(PRIOR_DIR, dtype=np.float32)); np.save(f"{OUT}/evid_dir.npy", np.array(EVID_DIR, dtype=np.float32))
np.save(f"{OUT}/shift_dir.npy", np.array(SHIFT_DIR, dtype=np.float32))       # NOT unit norm: mean (hist-none) difference, high count minus low count
say("\n[A4] geometry per layer. cos_prior_evid: alignment of count direction with grating-side direction (0 = orthogonal).")
say("     dprime_*: separation of right vs left grating along the none-run evidence axis, in the none and hist runs; ratio < 1 = evidence signal weakened by history")
say(A4[["layer", "cos_prior_evid", "dprime_none", "dprime_hist", "dprime_ratio", "evid_norm_ratio", "cos_prior_vs_shiftdir"]].iloc[::2].round(2).to_string(index=False))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(A1.layer, A1.count_corr_hist, label="hist run"); ax[0].plot(A1.layer, A1.count_corr_none, label="none run (control)"); ax[0].set_title("A1 count direction tracks count"); ax[0].legend(); ax[0].set_xlabel("layer")
ax[1].plot(A1.layer, A1.partial_corr_shift); ax[1].fill_between(A1.layer, A1.pc_lo, A1.pc_hi, alpha=.25); ax[1].axhline(0, c="gray", lw=.5); ax[1].set_title("A2 partial corr: z vs shift | cells+count"); ax[1].set_xlabel("layer")
ax[2].plot(A4.layer, A4.dprime_ratio, label="d' ratio hist/none"); ax[2].plot(A4.layer, A4.cos_prior_evid, label="cos(count dir, evidence dir)"); ax[2].axhline(0.55, c="gray", ls=":", lw=.8); ax[2].set_title("A4 geometry"); ax[2].legend(fontsize=8); ax[2].set_xlabel("layer")
plt.tight_layout(); plt.savefig(f"{OUT}/a_overview.png", dpi=130); plt.close()

In [ ]:
# ===================== SUMMARY: paste everything below back to Claude =====================
best = A1.loc[A1.partial_corr_shift.idxmax()]
say("\n=== HEADLINE ===")
say(f"count direction tracks the count best at layer {int(A1.loc[A1.count_corr_hist.idxmax(),'layer'])} (corr {A1.count_corr_hist.max():.2f}); control (no-history run) max corr {A1.count_corr_none.max():.2f}")
say(f"strongest 'explains shift beyond count' at layer {int(best.layer)}: partial corr {best.partial_corr_shift:+.2f} [{best.pc_lo:+.2f},{best.pc_hi:+.2f}], out-of-fold R2 gain {best.dR2:+.3f}")
say(f"layers where the partial-corr CI excludes 0: {A1.layer[(A1.pc_lo > 0) | (A1.pc_hi < 0)].tolist()}")
say(f"d' ratio hist/none: layer 4: {A4.dprime_ratio[4]:.2f} | 12: {A4.dprime_ratio[12]:.2f} | 20: {A4.dprime_ratio[20]:.2f} | 32: {A4.dprime_ratio[NL-1]:.2f}")
open(f"{OUT}/A_summary.txt", "w").write("\n".join(SUMMARY))
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"\nzip written: {OUT}.zip  (summary, csvs, plots, and the three direction files .npy; all small)")